<a href="https://colab.research.google.com/github/ali-liyakat/PyTorch_for_Deep_Learning/blob/main/02_pytorch_autograd.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## PyTorch Autograd

In [1]:
def dy_dx(x):
  return 2*x

In [2]:
dy_dx(3)

6

In [3]:
import torch

In [4]:
x = torch.tensor(3.0, requires_grad=True)

In [5]:
y = x**2

In [6]:
x

tensor(3., requires_grad=True)

In [7]:
y

tensor(9., grad_fn=<PowBackward0>)

In [8]:
y.backward()

In [9]:
x.grad

tensor(6.)

In [10]:
import math

In [11]:
def dz_dx(x):
  return 2 * x * math.cos(x**2)

In [12]:
dz_dx(3)

-5.466781571308061

In [13]:
x = torch.tensor(3.0, requires_grad=True)

In [14]:
y = x ** 2

In [15]:
z = torch.sin(y)

In [16]:
x

tensor(3., requires_grad=True)

In [17]:
y

tensor(9., grad_fn=<PowBackward0>)

In [18]:
z

tensor(0.4121, grad_fn=<SinBackward0>)

In [19]:
z.backward()

In [20]:
x.grad

tensor(-5.4668)

**Perceptron**

Manually

In [21]:
# Inputs
x = torch.tensor(6.7)  # Input feature
y = torch.tensor(0.0)  # True label (binary)

w = torch.tensor(1.0)  # Weight
b = torch.tensor(0.0)  # Bias

In [22]:
# Binary Cross-Entropy Loss for scalar
def binary_cross_entropy_loss(prediction, target):
    epsilon = 1e-8  # To prevent log(0)
    prediction = torch.clamp(prediction, epsilon, 1 - epsilon)
    return -(target * torch.log(prediction) + (1 - target) * torch.log(1 - prediction))

In [23]:
# Forward pass
z = w * x + b  # Weighted sum (linear part)
y_pred = torch.sigmoid(z)  # Predicted probability

# Compute binary cross-entropy loss
loss = binary_cross_entropy_loss(y_pred, y)

In [24]:
loss

tensor(6.7012)

In [25]:
# Derivatives:
# 1. dL/d(y_pred): Loss with respect to the prediction (y_pred)
dloss_dy_pred = (y_pred - y)/(y_pred*(1-y_pred))

# 2. dy_pred/dz: Prediction (y_pred) with respect to z (sigmoid derivative)
dy_pred_dz = y_pred * (1 - y_pred)

# 3. dz/dw and dz/db: z with respect to w and b
dz_dw = x  # dz/dw = x
dz_db = 1  # dz/db = 1 (bias contributes directly to z)

dL_dw = dloss_dy_pred * dy_pred_dz * dz_dw
dL_db = dloss_dy_pred * dy_pred_dz * dz_db

In [26]:
print(f"Manual Gradient of loss w.r.t weight (dw): {dL_dw}")
print(f"Manual Gradient of loss w.r.t bias (db): {dL_db}")

Manual Gradient of loss w.r.t weight (dw): 6.691762447357178
Manual Gradient of loss w.r.t bias (db): 0.998770534992218


**Using Autograd**

In [27]:
x = torch.tensor(6.7)
y = torch.tensor(0.0)

In [28]:
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.0, requires_grad=True)

In [29]:
w

tensor(1., requires_grad=True)

In [30]:
b

tensor(0., requires_grad=True)

In [31]:
z = w*x + b
z

tensor(6.7000, grad_fn=<AddBackward0>)

In [32]:
y_pred = torch.sigmoid(z)
y_pred

tensor(0.9988, grad_fn=<SigmoidBackward0>)

In [33]:
loss = binary_cross_entropy_loss(y_pred, y)
loss

tensor(6.7012, grad_fn=<NegBackward0>)

In [34]:
loss.backward()

In [35]:
print(w.grad)
print(b.grad)

tensor(6.6918)
tensor(0.9988)


**Vector Inputs**

In [36]:
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)

In [37]:
x

tensor([1., 2., 3.], requires_grad=True)

In [38]:
y = (x**2).mean()
y

tensor(4.6667, grad_fn=<MeanBackward0>)

In [39]:
y.backward()

In [40]:
x.grad

tensor([0.6667, 1.3333, 2.0000])

**Clearing Grad**

In [41]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [42]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [43]:
y.backward()

In [44]:
x.grad

tensor(4.)

In [45]:
x.grad.zero_()

tensor(0.)

**Disable Gradient Tracking**

In [46]:
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [47]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [48]:
y.backward()

In [49]:
x.grad

tensor(4.)

In [50]:
# option 1 - requires_grad_(False)

x.requires_grad_(False)
x.grad

tensor(4.)

In [51]:
x

tensor(2.)

In [52]:
y = x ** 2
y

tensor(4.)

In [53]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn

In [54]:
# option 2 - detach()
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [55]:
z = x.detach()
z

tensor(2.)

In [56]:
y = x ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [57]:
y1 = z ** 2
y

tensor(4., grad_fn=<PowBackward0>)

In [58]:
# option 3 - torch.no_grad()
x = torch.tensor(2.0, requires_grad=True)
x

tensor(2., requires_grad=True)

In [59]:
with torch.no_grad():
  y = x ** 2

In [60]:
y

tensor(4.)

In [61]:
y.backward()

RuntimeError: element 0 of tensors does not require grad and does not have a grad_fn